### 1. Setup Inicial
 

In [1]:
# Importação das bibliotecas

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configurações de exibição

pd.set_option('display.max_columns', None)   
pd.set_option('display.float_format', '{:.2f}'.format)

print('Bibliotecas carregadas!')


Bibliotecas carregadas!


In [2]:
# Carregamento dos dados brutos

df_raw = pd.read_csv('../data/Base Varejo.csv', sep=';')

print('Dataframe bruto')
print(f'   Linhas x Colunas : {df_raw.shape[0]} x {df_raw.shape[1]}')
print(f'   Nulos totais     : {df_raw.isnull().sum().sum()}')
print(f'   Duplicatas       : {df_raw.duplicated().sum()}')

# Primeiras visualizações 

print('\n--- Informações do Dataframe ---')
display(df_raw.info())

print('\n--- Primeiras linhas do Dataframe ---')
display(df_raw.head(10))


Dataframe bruto
   Linhas x Colunas : 830000 x 14
   Nulos totais     : 3320000
   Duplicatas       : 96553

--- Informações do Dataframe ---
<class 'pandas.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  str    
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  str    
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  str    
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  str    
 9   PR_NOME      830000 non-null  str    
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), str(5)
memory usage: 88.7 MB


None


--- Primeiras linhas do Dataframe ---


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN
5,01/02/2019,1000,534,M,4,1,C,187,HIGIENE,HASTES FLEXIVEIS,NaN,NaN,NaN,NaN
6,01/02/2019,1000,534,M,4,1,C,163,ALIMENTOS,MORTADELA,NaN,NaN,NaN,NaN
7,01/02/2019,1000,534,M,4,1,C,11,ALIMENTOS,AZEITE,NaN,NaN,NaN,NaN
8,01/02/2019,1000,534,M,4,1,C,95,LIMPEZA,AMACIANTE,NaN,NaN,NaN,NaN
9,01/02/2019,1000,534,M,4,1,C,198,BEBIDAS,ENERGETICO,NaN,NaN,NaN,NaN


#### Insigts

- A base está estruturada de forma transacional, pois existem valores nas colunas de ID de compra (`CO_ID`) e ID do cliente (`CL_ID`) que se repetem, tendo variação no ID do produto (`PR_ID`). Isso demostra uma jornada de compra.

- As 4 últimas colunas vieram vazias. Eliminação necessária.

- Coluna de DATA precisa ser alterada para o formato `datetime`

### 2. Transformações

In [4]:
# Primeiro, copiar o dataframe para as transformações
df_limpo = df_raw.copy()

# Eliminar as últimas quatro colunas que estão totalmente vazias
df_limpo = df_limpo.dropna(how='all', axis=1)

# Converter a coluna 'DATA' para datetime
df_limpo['DATA'] = pd.to_datetime(df_limpo['DATA'], format='%d/%m/%Y')

# Visualização das mudanças
print('\n--- Dataframe após tratamentos iniciais ---')
display(df_limpo.head())


--- Dataframe após tratamentos iniciais ---


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,2019-02-01,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


In [8]:
# Renomear os nomes das colunas para melhor visualização
df_limpo = df_limpo.rename(columns={
    'DATA': 'data_venda',
    'CO_ID': 'id_cupom',
    'CL_ID': 'id_cliente',
    'CL_GENERO': 'genero_cliente',
    'CL_EC': 'estado_civil_cliente',
    'CL_FHL': 'faixa_filhos_cliente',
    'CL_SEG': 'segmentacao_cliente',
    'PR_ID': 'id_produto',
    'PR_CAT': 'categoria_produto',
    'PR_NOME': 'nome_produto'
})

# Renomear os valores das colunas 'genero_cliente'
df_limpo['genero_cliente'] = df_limpo['genero_cliente'].replace({'M': 'Masculino', 'F': 'Feminino'})
print('--- Valores da coluna após o tratamento ---')
print(df_limpo['genero_cliente'].unique())

print('\n--- Dataframe após renomeações ---')
display(df_limpo.head(10))

--- Valores da coluna após o tratamento ---
<StringArray>
['Masculino', 'Feminino']
Length: 2, dtype: str

--- Dataframe após renomeações ---


,data_venda,id_cupom,id_cliente,genero_cliente,estado_civil_cliente,faixa_filhos_cliente,segmentacao_cliente,id_produto,categoria_produto,nome_produto
0,2019-02-01,1000,534,Masculino,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534,Masculino,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534,Masculino,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534,Masculino,4,1,C,4,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534,Masculino,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO
5,2019-02-01,1000,534,Masculino,4,1,C,187,HIGIENE,HASTES FLEXIVEIS
6,2019-02-01,1000,534,Masculino,4,1,C,163,ALIMENTOS,MORTADELA
7,2019-02-01,1000,534,Masculino,4,1,C,11,ALIMENTOS,AZEITE
8,2019-02-01,1000,534,Masculino,4,1,C,95,LIMPEZA,AMACIANTE
9,2019-02-01,1000,534,Masculino,4,1,C,198,BEBIDAS,ENERGETICO


### 3. Limpeza de Nulos e Duplicatas

In [17]:
# Verificação de nulos por colunas
nulos = df_limpo.isnull().sum()
pct = (nulos / len(df_limpo) * 100).round(1)

print(f'\n--- Nulos por coluna ---')
print(pct)


--- Nulos por coluna ---
data_venda             0.00
id_cupom               0.00
id_cliente             0.00
genero_cliente         0.00
estado_civil_cliente   0.00
faixa_filhos_cliente   0.00
segmentacao_cliente    0.00
id_produto             0.00
categoria_produto      0.00
nome_produto           0.00
dtype: float64


In [ ]:
# Verificação de duplicatas por coluna

print(f'\nDuplicatas: {df_limpo.duplicated().sum()}')

#Verificas se as duplicadas são reais ou 
df_limpo[df_limpo.duplicated(keep=False)].head(10)


Duplicatas: 96553


,data_venda,id_cupom,id_cliente,genero_cliente,estado_civil_cliente,faixa_filhos_cliente,segmentacao_cliente,id_produto,categoria_produto,nome_produto
3,2019-02-01,1000,534,Masculino,4,1,C,4,ALIMENTOS,ABACAXI
7,2019-02-01,1000,534,Masculino,4,1,C,11,ALIMENTOS,AZEITE
14,2019-02-01,1000,534,Masculino,4,1,C,13,ALIMENTOS,BANANA
15,2019-02-01,1000,534,Masculino,4,1,C,218,ALIMENTOS,BIFE DE COXAO MOLE
19,2019-02-01,1000,534,Masculino,4,1,C,13,ALIMENTOS,BANANA
22,2019-02-01,1000,534,Masculino,4,1,C,69,BEBIDAS,REFRIGERANTE LIMaO
34,2019-02-01,1000,534,Masculino,4,1,C,225,ALIMENTOS,ATUM
40,2019-02-01,1000,534,Masculino,4,1,C,4,ALIMENTOS,ABACAXI
46,2019-02-01,1000,534,Masculino,4,1,C,11,ALIMENTOS,AZEITE
49,2019-02-01,1000,534,Masculino,4,1,C,69,BEBIDAS,REFRIGERANTE LIMaO


#### Insigths

- Ao analisar as duplicadas, conclui que não são duplicatas reais, mas sim o registo do mesmo item para o mesmo cliente no mesmo cupom. Com isso, decidi manter todos os valores e analisar melhor agrupando na etapa de estatística descritiva

 

### 4. Estatística Descritiva

In [21]:
df_limpo.describe()

,data_venda,id_cupom,id_cliente,estado_civil_cliente,faixa_filhos_cliente,id_produto
count,830000,830000.00,830000.00,830000.00,830000.00,830000.00
mean,2020-12-06 14:55:46.421204,460045.09,499.60,2.60,1.15,115.05
min,2019-01-04 00:00:00,1000.00,1.00,1.00,0.00,1.00
25%,2020-01-09 00:00:00,233117.00,254.00,2.00,0.00,58.00
50%,2020-12-27 00:00:00,456517.00,498.00,3.00,0.00,115.00
75%,2021-10-20 00:00:00,690132.00,746.00,4.00,2.00,172.00
max,2022-12-08 00:00:00,919822.00,1000.00,5.00,4.00,229.00
std,NaN,265465.25,287.57,1.17,1.42,66.13


In [22]:
df_limpo['nome_produto'].value_counts().head(10)

nome_produto
PRESUNTO COZIDO       14381
SARDINHA               7490
GEL                    7399
BANANA                 7385
DESENGORDURANTE        7378
MODELADOR              7363
BIFE DE COXAO MOLE     7355
REMOVEDOR              7350
ESCOVA DE DENTE        7346
PAPINHA INFANTIL       7346
Name: count, dtype: int64

### 5. Exploração